In [0]:

source_path = "abfss://healthcare@ecommerceunifiedproject.dfs.core.windows.net/staging/hospitals/"
checkpoint_path = "abfss://healthcare@ecommerceunifiedproject.dfs.core.windows.net/bronze/hospitals_raw/checkpoint/"
schema_location = "abfss://healthcare@ecommerceunifiedproject.dfs.core.windows.net/bronze/hospitals_raw/schema/"

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.maxFilesPerTrigger", 1)
    .option("cloudFiles.schemaLocation", schema_location)
    .load(source_path)
)

query = (
    df.drop("_rescued_data")
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("healthcare_dev.bronze.hospitals_raw")
)

query.awaitTermination()
